# 01 - CAPM et Fama-French sur un seul actif

Objectif : comprendre le pipeline complet sur **un** actif avant de passer à plusieurs.

1. Charger les facteurs et les prix
2. Calculer le rendement excédentaire
3. Estimer CAPM, FF3 et FF5
4. Lire les résultats

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from factor_analysis import load_ff_factors, get_monthly_returns, fit_factor_model, compare_models
from factor_analysis.regression import FF3

## 1. Données

In [ ]:
factors = load_ff_factors()        # téléchargé une fois, puis mis en cache
factors.tail()

In [ ]:
TICKER = "AAPL"
returns = get_monthly_returns([TICKER], start="2005-01-01")
returns.tail()

## 2. Rendement excédentaire

On soustrait le taux sans risque `RF` au rendement de l'actif, puis on aligne les dates avec les facteurs.

In [ ]:
data = returns[[TICKER]].join(factors, how="inner").dropna()
excess = data[TICKER] - data["RF"]
print(f"{len(data)} mois, de {data.index[0]} à {data.index[-1]}")

## 3. Régression FF3

Interprétation :
- **alpha** : rendement moyen mensuel non expliqué par les facteurs ;
- **beta Mkt-RF** : sensibilité au marché ;
- **SMB** : exposition à la taille (petites contre grandes capitalisations) ;
- **HML** : exposition à la valeur (value contre growth).

In [ ]:
res = fit_factor_model(excess, factors, FF3, hac_lags=6)
print(res.summary())

## 4. Comparaison des modèles

Attention : un R² ajusté plus élevé ne veut pas dire que le modèle est « vrai ». Regarde aussi BIC, et surtout la significativité de l'alpha.

In [ ]:
compare_models(excess, factors).round(4)

## À toi de jouer

- Change `TICKER` (une valeur « growth », une valeur « value », une petite capitalisation). Les bêtas SMB et HML changent-ils comme tu l'attends ?
- L'alpha est-il significatif (|t| > 2) ? Pourquoi faut-il être prudent sur un seul actif ?
- Compare les erreurs standard avec `hac_lags=None` (OLS classique) et `hac_lags=6`.